# GEM-XAI news sentiment features

1. **Runtime → Change runtime type → T4 GPU**
2. Paste your Guardian key in the cell marked **KEY**
3. **Runtime → Run all**

The output `daily_sentiment.csv` is saved to Google Drive (`MyDrive/gemxai_news`) and downloaded at the end.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%mkdir -p /content/drive/MyDrive/gemxai_news
%cd /content/drive/MyDrive/gemxai_news
!pip install -q pandas_market_calendars

In [ ]:
# KEY: paste your free Guardian Open Platform key (https://open-platform.theguardian.com/access/)
GUARDIAN_KEY = "PASTE_YOUR_KEY_HERE"

In [ ]:
%%writefile 1_download_news.py
"""
Step 1 - Download financial news headlines for 2010-01-01 .. 2025-06-30.

Run this on YOUR machine or AWS (it needs internet). Two free sources, pick one
(Guardian is recommended: one consistent source for the whole period, generous limits).

  Guardian Open Platform  - free developer key: https://open-platform.theguardian.com/access/
  NYT Archive API         - free key:          https://developer.nytimes.com/get-started

Usage:
  pip install requests pandas
  python 1_download_news.py --source guardian --key YOUR_KEY
  python 1_download_news.py --source nyt      --key YOUR_KEY

Output: headlines_<source>.csv with columns: published_utc, headline, section, source
The script resumes safely if interrupted (it skips months already saved).
"""
import argparse, os, time, json
import requests
import pandas as pd

START, END = (2010, 1), (2025, 6)


def months():
    y, m = START
    while (y, m) <= END:
        yield y, m
        m += 1
        if m == 13:
            y, m = y + 1, 1


def guardian_month(y, m, key):
    rows, page, pages = [], 1, 1
    last = pd.Timestamp(y, m, 1) + pd.offsets.MonthEnd(0)
    while page <= pages:
        r = requests.get("https://content.guardianapis.com/search", params={
            "section": "business", "from-date": f"{y}-{m:02d}-01", "to-date": last.strftime("%Y-%m-%d"),
            "page-size": 200, "page": page, "order-by": "oldest", "api-key": key}, timeout=60)
        r.raise_for_status()
        resp = r.json()["response"]
        pages = resp.get("pages", 1)
        for it in resp.get("results", []):
            rows.append({"published_utc": it["webPublicationDate"], "headline": it["webTitle"],
                         "section": it.get("sectionName", ""), "source": "guardian"})
        page += 1
        time.sleep(0.2)
    return rows


NYT_DESKS = {"Business", "BusinessDay", "Business Day", "SundayBusiness", "Financial", "DealBook"}


def nyt_month(y, m, key):
    r = requests.get(f"https://api.nytimes.com/svc/archive/v1/{y}/{m}.json", params={"api-key": key}, timeout=120)
    r.raise_for_status()
    rows = []
    for d in r.json()["response"]["docs"]:
        sec, desk = d.get("section_name") or "", d.get("news_desk") or ""
        if sec in NYT_DESKS or desk in NYT_DESKS:
            h = (d.get("headline") or {}).get("main") or ""
            if h:
                rows.append({"published_utc": d["pub_date"], "headline": h, "section": sec or desk, "source": "nyt"})
    time.sleep(12.5)  # NYT limit: 5 requests per minute
    return rows


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--source", choices=["guardian", "nyt"], required=True)
    ap.add_argument("--key", required=True)
    a = ap.parse_args()
    out = f"headlines_{a.source}.csv"
    done_file = out + ".done.json"
    done = set(json.load(open(done_file))) if os.path.exists(done_file) else set()
    fetch = guardian_month if a.source == "guardian" else nyt_month
    for y, m in months():
        tag = f"{y}-{m:02d}"
        if tag in done:
            continue
        for attempt in range(5):
            try:
                rows = fetch(y, m, a.key)
                break
            except Exception as e:
                print(tag, "retry", attempt + 1, e)
                time.sleep(30)
        else:
            raise SystemExit(f"Failed on {tag}; re-run the script to resume.")
        pd.DataFrame(rows).to_csv(out, mode="a", header=not os.path.exists(out), index=False)
        done.add(tag)
        json.dump(sorted(done), open(done_file, "w"))
        print(tag, len(rows), "headlines")
    df = pd.read_csv(out).drop_duplicates(["published_utc", "headline"])
    df.to_csv(out, index=False)
    print("Total headlines:", len(df))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile 2_finbert_daily_features.py
"""
Step 2 - Score headlines with FinBERT and build the four daily sentiment features.

Run on AWS (GPU recommended; CPU works but is slower). Needs internet once to download
the ProsusAI/finbert model from Hugging Face.

Usage:
  pip install torch transformers pandas pandas_market_calendars
  python 2_finbert_daily_features.py --headlines headlines_guardian.csv --out daily_sentiment.csv

Leakage rule (same as the thesis): news published at or after the 16:00 New York close
is assigned to the NEXT trading session, so day t only uses news known by the close of day t.

Output columns (one row per NYSE trading day 2010-01-04 .. 2025-06-30):
  date, mean_pos, mean_neg, net_sentiment, sentiment_std, n_headlines
Days with no headlines get zeros and n_headlines = 0.
"""
import argparse
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import pandas_market_calendars as mcal


def trading_days():
    nyse = mcal.get_calendar("NYSE")
    sched = nyse.schedule(start_date="2009-12-01", end_date="2025-07-15")
    return pd.DatetimeIndex(sched.index.date)


def assign_session(ts_utc, days):
    ny = ts_utc.tz_convert("America/New_York")
    d = pd.DatetimeIndex(ny.date)
    after_close = np.asarray(ny.hour >= 16)
    pos = days.searchsorted(d, side="left")            # first trading day >= calendar date
    same = (pos < len(days)) & (days[np.minimum(pos, len(days) - 1)] == d)
    pos = np.where(same & after_close, pos + 1, pos)    # after close on a trading day -> next session
    pos = np.minimum(pos, len(days) - 1)
    return days[pos]


def score(headlines, batch=64):
    name = "ProsusAI/finbert"
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForSequenceClassification.from_pretrained(name).eval()
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(dev)
    lab = {v.lower(): k for k, v in model.config.id2label.items()}
    out = np.zeros((len(headlines), 3), dtype=np.float32)
    with torch.no_grad():
        for i in range(0, len(headlines), batch):
            enc = tok(headlines[i:i + batch], padding=True, truncation=True, max_length=64, return_tensors="pt").to(dev)
            p = torch.softmax(model(**enc).logits, dim=-1).cpu().numpy()
            out[i:i + batch, 0] = p[:, lab["positive"]]
            out[i:i + batch, 1] = p[:, lab["negative"]]
            out[i:i + batch, 2] = p[:, lab["neutral"]]
            if (i // batch) % 200 == 0:
                print(f"scored {i}/{len(headlines)}")
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--headlines", required=True)
    ap.add_argument("--out", default="daily_sentiment.csv")
    ap.add_argument("--max_per_day", type=int, default=0, help="optional cap per session for speed (0 = no cap)")
    a = ap.parse_args()

    df = pd.read_csv(a.headlines).dropna(subset=["headline"])
    df["published_utc"] = pd.to_datetime(df["published_utc"], utc=True)
    days = trading_days()
    df["date"] = assign_session(pd.DatetimeIndex(df["published_utc"]), days)
    df = df[(df["date"] >= "2010-01-01") & (df["date"] <= "2025-06-30")]
    if a.max_per_day:
        df = df.groupby("date", group_keys=False).apply(lambda g: g.sample(min(len(g), a.max_per_day), random_state=42))
    print("Headlines to score:", len(df))

    p = score(df["headline"].astype(str).tolist())
    df["pos"], df["neg"] = p[:, 0], p[:, 1]
    df["net"] = df["pos"] - df["neg"]
    daily = df.groupby("date").agg(mean_pos=("pos", "mean"), mean_neg=("neg", "mean"),
                                   net_sentiment=("net", "mean"), sentiment_std=("net", "std"),
                                   n_headlines=("net", "size"))
    daily["sentiment_std"] = daily["sentiment_std"].fillna(0.0)
    full = pd.DataFrame(index=days[(days >= "2010-01-01") & (days <= "2025-06-30")])
    full = full.join(daily).fillna({"mean_pos": 0, "mean_neg": 0, "net_sentiment": 0, "sentiment_std": 0, "n_headlines": 0})
    full.index.name = "date"
    full.to_csv(a.out)
    cov = (full["n_headlines"] > 0).mean()
    print(f"Saved {a.out}: {len(full)} trading days, {cov:.1%} with at least one headline")
    print(full.groupby(full.index.year)["n_headlines"].agg(["mean", "min"]))


if __name__ == "__main__":
    main()


In [ ]:
import torch; print('GPU available:', torch.cuda.is_available())
!python 1_download_news.py --source guardian --key {GUARDIAN_KEY}

In [ ]:
!python 2_finbert_daily_features.py --headlines headlines_guardian.csv --out daily_sentiment.csv

In [ ]:
import pandas as pd
from google.colab import files
d = pd.read_csv('daily_sentiment.csv')
print(d.shape)
print(d.describe())
files.download('daily_sentiment.csv')